# Runeterra Lore RAG v2 (Expanded with Web Lore & Portable Paths)
Changes compared to v1:
1. **Dynamic Environment & Paths**: Automatically detects **Google Colab** vs **Local Environment** (Windows/Linux), loading PDFs from `data/books/` or workspace root and settings from `.env`.
2. **Web Lore Expansion**: Ingests online lore (champion bios from Riot Data Dragon, Universe short stories and region lore from League Wiki) from `data/web_lore/`.
3. **Chunking**: ~320-word chunks with 50-word overlap that can span page boundaries.
4. **Hybrid retrieval**: BM25 + dense embeddings (`BAAI/bge-m3`), fused with Reciprocal Rank Fusion (RRF).
5. **Reranking**: Cross-encoder (`BAAI/bge-reranker-v2-m3`) re-scores the top fused candidates.
6. **Neighbor chunks**: The best hits pull in adjacent chunks, merged into continuous context.

Use a **GPU runtime** if available. Embeddings are cached automatically.

## 1. Environment Setup & Ingestion
Detects environment (Colab vs Local), discovers PDF books, runs OCR if needed, and caches pages.

In [ ]:
import io
import json
import os
import re
import sys
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

# ---------- Environment & Configuration ----------
IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    print("Detected Google Colab environment.")
    get_ipython().system("apt-get -qq install -y tesseract-ocr > /dev/null")
    get_ipython().system("pip install -q -U pypdf pymupdf pytesseract pillow google-genai gradio tqdm sentence-transformers python-dotenv")
    from google.colab import drive
    drive.mount('/content/drive')
    COLAB_BASE = Path("/content/drive/MyDrive/Lol")
    CACHE_DIR = str(COLAB_BASE / "cache")
else:
    print("Detected Local machine environment.")
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass
    BASE_DIR = Path.cwd()
    CACHE_DIR = str(BASE_DIR / "data" / "cache")

import fitz  # PyMuPDF
from PIL import Image
from pypdf import PdfReader
from tqdm import tqdm

os.makedirs(CACHE_DIR, exist_ok=True)

def find_pdf_paths():
    found = []
    # 1. Check local data/books and root
    books_dir = Path("data/books")
    if books_dir.exists():
        found.extend(list(books_dir.glob("*.pdf")))
    for p in Path(".").glob("*.pdf"):
        if p not in found:
            found.append(p)
    # 2. Check Google Colab Drive
    colab_dir = Path("/content/drive/MyDrive/Lol")
    if colab_dir.exists():
        for p in colab_dir.glob("*.pdf"):
            if p not in found:
                found.append(p)
    return sorted(list(set(str(p.resolve()) for p in found)))

PDF_PATHS = find_pdf_paths()
print(f"Found {len(PDF_PATHS)} book(s):")
for p in PDF_PATHS:
    print(f" - {p}")

MIN_CHARS = 80      # pages with less text than this go through OCR
OCR_DPI = 200       # higher = better accuracy, slower
BATCH = 16          # pages rendered per batch

def book_title(path):
    name = os.path.splitext(os.path.basename(path))[0]
    name = name.replace("OceanofPDF.com", "").replace("_", " ")
    return re.sub(r"\s+", " ", name).strip(" -")

def ocr_png(png_bytes):
    try:
        import pytesseract
        img = Image.open(io.BytesIO(png_bytes))
        return pytesseract.image_to_string(img, lang="eng")
    except Exception:
        return ""

def extract_book(path):
    title = book_title(path)
    cache_file = os.path.join(CACHE_DIR, re.sub(r"\W+", "_", title)[:60] + ".json")

    if os.path.exists(cache_file):
        with open(cache_file, encoding="utf-8") as f:
            print(f"Loaded from cache: {title}")
            return json.load(f)

    reader = PdfReader(path)
    total = len(reader.pages)

    texts = []
    for i in tqdm(range(total), desc=f"Text   | {title[:35]}"):
        try:
            texts.append(reader.pages[i].extract_text() or "")
        except Exception:
            texts.append("")

    need_ocr = [i for i, t in enumerate(texts) if len(t.strip()) < MIN_CHARS]
    print(f"{title}: {len(need_ocr)} of {total} pages need OCR")

    import shutil
    tess_available = bool(shutil.which('tesseract'))
    if need_ocr and tess_available:
        try:
            import pytesseract
            doc = fitz.open(path)
            with ThreadPoolExecutor(max_workers=2) as pool:
                for start in tqdm(range(0, len(need_ocr), BATCH), desc=f"OCR    | {title[:35]}"):
                    chunk = need_ocr[start:start + BATCH]
                    pngs = [doc[i].get_pixmap(dpi=OCR_DPI).tobytes("png") for i in chunk]
                    for i, text in zip(chunk, pool.map(ocr_png, pngs)):
                        if text:
                            texts[i] = text
            doc.close()
        except Exception as e:
            print(f"Notice: OCR skipped ({e}). Native text preserved.")
    elif need_ocr:
        print(f"Notice: Tesseract not found on system PATH. OCR skipped for {title}.")

    pages = [{"book": title, "page": i + 1, "text": t, "source_type": "book"}
             for i, t in enumerate(texts) if t.strip()]

    with open(cache_file, "w", encoding="utf-8") as f:
        json.dump(pages, f, ensure_ascii=False)
    return pages

PAGES = []
for path in PDF_PATHS:
    PAGES.extend(extract_book(path))

BOOKS = sorted({p["book"] for p in PAGES})
print(f"\n{len(PAGES)} pages loaded from {len(BOOKS)} book(s):")
for b in BOOKS:
    n = sum(1 for p in PAGES if p["book"] == b)
    print(f" - {b}: {n} pages")

## 2. Configuration, Chunking & Web Lore Integration
Pages are cut into overlapping word windows. Web lore documents from `data/web_lore/` are also loaded and chunked.

In [ ]:
import hashlib
import math
import time
import unicodedata
from collections import Counter, defaultdict
import numpy as np

# ---------- Configuration ----------
CHUNK_WORDS = int(os.getenv("CHUNK_WORDS", 320))          # ~420 tokens for English prose
OVERLAP_WORDS = int(os.getenv("OVERLAP_WORDS", 50))         # words shared by consecutive chunks (~65 tokens)
EMBED_MODEL = os.getenv("EMBED_MODEL", "BAAI/bge-m3")                # multilingual dense retriever
RERANK_MODEL = os.getenv("RERANK_MODEL", "BAAI/bge-reranker-v2-m3")   # multilingual cross-encoder
CANDIDATES = int(os.getenv("CANDIDATES", 50))            # hits kept per retriever before fusion
RERANK_POOL = int(os.getenv("RERANK_POOL", 20))           # fused candidates sent to the reranker
RRF_K = int(os.getenv("RRF_K", 60))                 # standard RRF constant
NEIGHBORS_FOR_TOP_N = int(os.getenv("NEIGHBORS_FOR_TOP_N", 2))    # top hits that also pull in adjacent chunks

HEADING_RE = re.compile(
    r"^\s*(?:CHAPTER|PART|PROLOGUE|EPILOGUE|INTERLUDE)\b.{0,60}$"
    r"|^\s*(?:Chapter|Part)\s+(?:\d+|[IVXLC]+)\b.{0,40}$"
    r"|^\s*#{1,3}\s+(.+)$"
)

def clean_page_text(text):
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)
    lines = [ln for ln in text.splitlines() if not re.fullmatch(r"\s*\d{1,4}\s*", ln)]
    return "\n".join(lines)

def build_chunks(pages, size=CHUNK_WORDS, overlap=OVERLAP_WORDS):
    step = size - overlap
    chunks = []
    for book in sorted({p["book"] for p in pages}):
        book_pages = sorted((p for p in pages if p["book"] == book), key=lambda p: p["page"])
        words, word_page, page_chapter = [], [], {}
        chapter = ""
        for p in book_pages:
            text = clean_page_text(p["text"])
            for ln in text.splitlines():
                if HEADING_RE.match(ln):
                    chapter = ln.strip("# \t")
            page_chapter[p["page"]] = chapter
            w = text.split()
            words.extend(w)
            word_page.extend([p["page"]] * len(w))

        for start in range(0, len(words), step):
            window = words[start:start + size]
            if start > 0 and len(window) <= overlap:
                break
            first_page = word_page[start]
            chunks.append({
                "id": len(chunks),
                "book": book,
                "text": " ".join(window),
                "page_start": first_page,
                "page_end": word_page[start + len(window) - 1],
                "chapter": page_chapter[first_page],
                "source_type": "book",
            })
    return chunks

def load_web_lore_chunks(web_lore_dir="data/web_lore", size=CHUNK_WORDS, overlap=OVERLAP_WORDS):
    target = Path(web_lore_dir)
    if not target.exists():
        return []
    step = size - overlap
    web_chunks = []
    for jf in target.rglob("*.json"):
        try:
            with open(jf, "r", encoding="utf-8") as f:
                doc = json.load(f)
        except Exception:
            continue
        text = doc.get("text", "")
        if not text.strip():
            continue
        category = doc.get("category", "web_lore")
        name = doc.get("name")
        raw_title = doc.get("title")
        if category == "champion_bio" and name:
            title = f"{name} - {raw_title}" if raw_title else name
        else:
            title = raw_title or name or jf.stem
        source_name = f"Web: {title}"
        words = text.split()
        if len(words) <= size:
            web_chunks.append({
                "book": source_name,
                "text": text,
                "page_start": 1,
                "page_end": 1,
                "chapter": f"Category: {category}",
                "source_type": "web",
            })
        else:
            for start in range(0, len(words), step):
                window = words[start:start + size]
                if start > 0 and len(window) <= overlap:
                    break
                part = (start // step) + 1
                web_chunks.append({
                    "book": source_name,
                    "text": " ".join(window),
                    "page_start": part,
                    "page_end": part,
                    "chapter": f"{title} (Part {part})",
                    "source_type": "web",
                })
    return web_chunks

CHUNKS = build_chunks(PAGES)
WEB_CHUNKS = load_web_lore_chunks()
if WEB_CHUNKS:
    print(f"Loaded {len(WEB_CHUNKS)} additional chunk(s) from Web Lore (data/web_lore/)")
    for wc in WEB_CHUNKS:
        wc["id"] = len(CHUNKS)
        CHUNKS.append(wc)

BOOK_IDS = defaultdict(set)
for c in CHUNKS:
    BOOK_IDS[c["book"]].add(c["id"])

print(f"\n{len(CHUNKS)} total chunks across {len(BOOK_IDS)} source(s):\n")
for b, ids in sorted(BOOK_IDS.items()):
    print(f" - {b}: {len(ids)} chunks")

## 3. Sparse retriever (BM25 over chunks)

In [ ]:
STOPWORDS = {
    "the", "and", "for", "with", "that", "this", "from", "was", "were", "are", "his",
    "her", "their", "its", "has", "have", "had", "who", "what", "which", "when",
    "where", "how", "why", "does", "did", "about", "into", "than", "then", "there",
    "but", "not", "you", "your", "she", "him", "they", "them", "been", "will", "would",
    "de", "do", "da", "dos", "das", "em", "no", "na", "nos", "nas", "por", "para", "com",
}

def normalize(text):
    text = unicodedata.normalize("NFD", text.lower())
    return "".join(c for c in text if unicodedata.category(c) != "Mn")

def tokenize(text):
    tokens = re.findall(r"[a-z0-9]+", normalize(text))
    return [t[:6] if len(t) > 6 else t
            for t in tokens if len(t) > 2 and t not in STOPWORDS]

class BM25Index:
    """BM25 with an inverted index. Repeated query terms count more (used for keyword boosting)."""

    def __init__(self, docs, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        tfs = [Counter(tokenize(d)) for d in docs]
        self.lens = [sum(c.values()) for c in tfs]
        self.avg_len = (sum(self.lens) / len(self.lens)) if self.lens else 1
        self.postings = defaultdict(list)
        for i, c in enumerate(tfs):
            for t, f in c.items():
                self.postings[t].append((i, f))
        n = len(docs)
        self.idf = {t: math.log(1 + (n - len(p) + 0.5) / (len(p) + 0.5))
                    for t, p in self.postings.items()}

    def search(self, query, top_k=50, allowed=None):
        scores = defaultdict(float)
        for term, qtf in Counter(tokenize(query)).items():
            idf = self.idf.get(term)
            if idf is None:
                continue
            for doc_id, f in self.postings[term]:
                if allowed is not None and doc_id not in allowed:
                    continue
                denom = f + self.k1 * (1 - self.b + self.b * self.lens[doc_id] / self.avg_len)
                scores[doc_id] += qtf * idf * f * (self.k1 + 1) / denom
        ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        return ranked[:top_k]

BM25 = BM25Index([c["text"] for c in CHUNKS])

## 4. Dense retriever (multilingual embeddings, cached)
The cache key hashes the model name and chunk contents, re-embedding automatically when modified.

In [ ]:
import torch
from sentence_transformers import CrossEncoder, SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cpu":
    print("Notice: running on CPU. For faster embedding, consider a GPU runtime.")

embedder = SentenceTransformer(EMBED_MODEL, device=device)
embedder.max_seq_length = 512
if device == "cuda":
    embedder.half()

def embed_texts(texts, is_query=False):
    prefix = ("query: " if is_query else "passage: ") if "e5" in EMBED_MODEL.lower() else ""
    vecs = embedder.encode(
        [prefix + t for t in texts],
        batch_size=32 if device == "cuda" else 8,
        normalize_embeddings=True,
        show_progress_bar=len(texts) > 64,
        convert_to_numpy=True,
    )
    return np.asarray(vecs, dtype=np.float32)

def load_or_build_embeddings(chunks):
    if not chunks:
        return np.zeros((0, 1024), dtype=np.float32)
    h = hashlib.sha1(EMBED_MODEL.encode())
    for c in chunks:
        h.update(c["text"].encode("utf-8"))
    path = os.path.join(CACHE_DIR, f"emb_{h.hexdigest()[:16]}.npy")
    if os.path.exists(path):
        print("Loaded embeddings from cache:", os.path.basename(path))
        return np.load(path)
    print(f"Embedding {len(chunks)} chunks (first run only)...")
    emb = embed_texts([c["text"] for c in chunks])
    np.save(path, emb)
    return emb

EMB = load_or_build_embeddings(CHUNKS)
print("Embeddings shape:", EMB.shape)

def dense_search(query, top_k=50, allowed=None):
    if len(EMB) == 0:
        return []
    q = embed_texts([query], is_query=True)[0]
    sims = EMB @ q
    if allowed is not None:
        mask = np.full(len(sims), -np.inf, dtype=np.float32)
        mask[list(allowed)] = 0.0
        sims = sims + mask
    top = np.argsort(-sims)[:top_k]
    return [(int(i), float(sims[i])) for i in top if np.isfinite(sims[i])]

## 5. Hybrid fusion, reranking and context building
`retrieve()` supports four modes: `bm25`, `dense`, `hybrid` (RRF) and `hybrid+rerank`.

In [ ]:
reranker = CrossEncoder(RERANK_MODEL, device=device, max_length=512)

MODES = ["bm25", "dense", "hybrid", "hybrid+rerank"]

def rrf_fuse(rankings, k=RRF_K):
    """Reciprocal Rank Fusion: each ranked list votes 1 / (k + rank) for its documents."""
    scores = defaultdict(float)
    for ranking in rankings:
        for rank, (doc_id, _) in enumerate(ranking, start=1):
            scores[doc_id] += 1.0 / (k + rank)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

def retrieve(question, keywords="", books=None, mode="hybrid+rerank", top_k=4):
    if mode not in MODES:
        raise ValueError(f"Unknown mode: {mode}")
    allowed = set().union(*(BOOK_IDS[b] for b in books)) if books else None

    bm25_hits = BM25.search(f"{question} {keywords} {keywords}", CANDIDATES, allowed)
    if mode == "bm25":
        return bm25_hits[:top_k]

    dense_hits = dense_search(question, CANDIDATES, allowed)
    if mode == "dense":
        return dense_hits[:top_k]

    rankings = [bm25_hits, dense_hits]
    if keywords:
        rankings.append(dense_search(keywords, CANDIDATES, allowed))
    fused = rrf_fuse(rankings)
    if mode == "hybrid":
        return fused[:top_k]

    pool = fused[:RERANK_POOL]
    if not pool:
        return []
    pairs = [(question, CHUNKS[cid]["text"]) for cid, _ in pool]
    scores = reranker.predict(pairs, batch_size=8, show_progress_bar=False)
    reranked = sorted(zip([cid for cid, _ in pool], map(float, scores)),
                      key=lambda x: x[1], reverse=True)
    return reranked[:top_k]

def build_context_blocks(hit_ids, neighbors_for=NEIGHBORS_FOR_TOP_N):
    selected = set(hit_ids)
    for cid in hit_ids[:neighbors_for]:
        for nid in (cid - 1, cid + 1):
            if 0 <= nid < len(CHUNKS) and CHUNKS[nid]["book"] == CHUNKS[cid]["book"]:
                selected.add(nid)

    runs, run = [], []
    for cid in sorted(selected):
        if run and cid == run[-1] + 1 and CHUNKS[cid]["book"] == CHUNKS[run[-1]]["book"]:
            run.append(cid)
        else:
            if run:
                runs.append(run)
            run = [cid]
    if run:
        runs.append(run)

    blocks = []
    for run in runs:
        parts = [CHUNKS[run[0]]["text"]]
        for cid in run[1:]:
            parts.append(" ".join(CHUNKS[cid]["text"].split()[OVERLAP_WORDS:]))
        first, last = CHUNKS[run[0]], CHUNKS[run[-1]]
        blocks.append({
            "book": first["book"],
            "chapter": first["chapter"],
            "page_start": first["page_start"],
            "page_end": last["page_end"],
            "text": " ".join(parts),
            "chunk_ids": run,
            "source_type": first.get("source_type", "book"),
        })
    return blocks

def page_label(block):
    if block.get("source_type") == "web":
        return "Web"
    a, b = block["page_start"], block["page_end"]
    return str(a) if a == b else f"{a}-{b}"

## 6. Question answering with Gemini

In [ ]:
from google import genai
from google.genai import errors

api_key = os.getenv("GEMINI_API_KEY")
if not api_key and IS_COLAB:
    try:
        from google.colab import userdata
        api_key = userdata.get("GEMINI_API_KEY")
    except Exception:
        pass

if not api_key:
    print("Warning: GEMINI_API_KEY not found. Set it in .env or Colab Secrets.")
    client = None
else:
    client = genai.Client(api_key=api_key)

MODELS = [os.getenv("GEMINI_MODEL", "gemini-3.5-flash-lite"), "gemini-3.5-flash"]
NOT_FOUND = "I could not find this information in the books or lore documents."

def call_model(prompt, max_attempts=3):
    if not client:
        return ""
    for model in MODELS:
        for attempt in range(1, max_attempts + 1):
            try:
                r = client.models.generate_content(model=model, contents=prompt)
                return (r.text or "").strip()
            except errors.ServerError:
                time.sleep(2 * attempt)
            except errors.ClientError:
                break
    return ""

def expand_query(question):
    prompt = f"""The user asks a question about League of Legends lore (Runeterra, Ruination, Shadow Isles, champions, regions).
Write 8 to 12 English search keywords, comma-separated, to find the answer in the lore documents.
Include proper nouns (characters, places, events) exactly as they would be spelled in English.
Output ONLY the keywords.

Question: {question}"""
    return call_model(prompt, max_attempts=1)

def answer_question(question, selected_books, top_k, mode="hybrid+rerank"):
    question = (question or "").strip()
    if not question:
        return "Type a question first.", []

    books = selected_books or list(BOOK_IDS.keys())
    keywords = expand_query(question) if mode != "dense" else ""
    hits = retrieve(question, keywords, books, mode, int(top_k))
    if not hits:
        return NOT_FOUND, []

    blocks = build_context_blocks([cid for cid, _ in hits])
    parts = []
    for n, b in enumerate(blocks, start=1):
        chapter = f" | Chapter/Section: {b['chapter']}" if b["chapter"] else ""
        pages = f" | Pages: {page_label(b)}"
        parts.append(f"[S{n}] Source: {b['book']}{chapter}{pages}\n{b['text']}")
    context = "\n\n".join(parts)

    prompt = f"""You are a lore assistant for League of Legends books and universe documents.
Rules:
1. Answer ONLY using the context below. No outside knowledge.
2. Reply in the same language as the question (Portuguese if asked in Portuguese).
3. Paraphrase in your own words. Never copy long passages; at most a very short phrase.
4. Output EXACTLY this format:
ANSWER: <answer>
USED: <source ids you used, comma-separated, e.g. S1,S3>
5. If the answer is not in the context, output:
ANSWER: {NOT_FOUND}
USED: none

--- CONTEXT ---
{context}

--- QUESTION ---
{question}
"""
    raw = call_model(prompt)
    if not raw:
        return "No response from the API. Check your GEMINI_API_KEY in .env.", []

    m_ans = re.search(r"ANSWER:\s*(.*?)\s*(?:USED:|$)", raw, re.S)
    m_used = re.search(r"USED:\s*(.*)", raw)
    answer = m_ans.group(1).strip() if m_ans else raw

    if NOT_FOUND in answer:
        return NOT_FOUND, []

    used_ids = set(re.findall(r"S(\d+)", m_used.group(1))) if m_used else set()
    sources = [[b["book"], page_label(b), b["chapter"]]
               for n, b in enumerate(blocks, start=1) if str(n) in used_ids]
    return answer, sources

## 7. Gradio UI

In [ ]:
import gradio as gr

EXAMPLES = [
    ["Who is the Ruined King and how was he created?"],
    ["O que é a Ruína e como ela afetou as Ilhas das Sombras?"],
    ["Quais são as principais regiões de Runeterra e suas características?"],
    ["What is the speed of a Boeing 747?"],  # control question: should return not found
]

def ui_ask(question, selected_books, top_k, mode):
    return answer_question(question, selected_books, top_k, mode)

all_sources = sorted(list(BOOK_IDS.keys()))

with gr.Blocks(title="Runeterra Lore Q&A") as demo:
    gr.Markdown("# ⚔️ Runeterra Lore Q&A\nAsk questions about League of Legends books and Web Lore. "
                "Answers come strictly from the loaded documents, with sources and citations.")

    with gr.Row():
        with gr.Column(scale=2):
            question = gr.Textbox(label="Question", placeholder="Ask about characters, regions, events...", lines=2)
            with gr.Accordion("Options", open=False):
                books_in = gr.CheckboxGroup(choices=all_sources, value=all_sources, label="Sources to search")
                top_k = gr.Slider(1, 8, value=4, step=1, label="Chunks used as context (neighbors added on top)")
                mode_in = gr.Dropdown(choices=MODES, value="hybrid+rerank", label="Retrieval mode")
            ask_btn = gr.Button("Ask", variant="primary")
            gr.Examples(examples=EXAMPLES, inputs=[question])
        with gr.Column(scale=3):
            answer_out = gr.Textbox(label="Answer", lines=10, interactive=False)
            sources_out = gr.Dataframe(headers=["Source", "Pages / Type", "Chapter / Section"],
                                       datatype=["str", "str", "str"],
                                       label="Sources", interactive=False, wrap=True)

    ask_btn.click(ui_ask, inputs=[question, books_in, top_k, mode_in], outputs=[answer_out, sources_out])
    question.submit(ui_ask, inputs=[question, books_in, top_k, mode_in], outputs=[answer_out, sources_out])

demo.launch(share=False, debug=False)